# Deploy NVIDIA Parakeet Realtime EOU 120M on Vertex AI — Triton Inference Server

This notebook deploys the [NVIDIA Parakeet Realtime EOU 120M](https://huggingface.co/nvidia/parakeet_realtime_eou_120m-v1) model on **Google Cloud Vertex AI** using **NVIDIA Triton Inference Server** with its Python Backend for optimized concurrent inference.

| | |
|---|---|
| **Task** | Streaming ASR with End-of-Utterance (EOU) detection |
| **Architecture** | FastConformer-RNNT (Cache-Aware Streaming) |
| **Parameters** | 120 M |
| **Language** | English |
| **Input** | 16 kHz mono WAV audio |
| **Output** | Transcription with `<EOU>` tokens marking utterance boundaries |
| **Inference Server** | NVIDIA Triton (Python Backend + dynamic batching) |
| **License** | NVIDIA Open Model License |

### Why Triton over Flask+gunicorn?

| Feature | Flask+gunicorn | Triton Python Backend |
|---------|---------------|----------------------|
| Batching | None (sequential) | Dynamic batching (up to 8 requests) |
| GPU utilization | One request at a time | Batched inference — higher throughput |
| Concurrency | 1 worker, 2 threads | Triton scheduler + concurrent queue |
| Scalability | Manual | Built-in model instance management |

**What this notebook does:**
1. Builds a custom container with Triton Inference Server + NeMo model
2. Pushes it to Google Artifact Registry via Cloud Build
3. Deploys the model to a Vertex AI Endpoint on an NVIDIA T4 GPU
4. Sends sample audio for transcription and shows results
5. Cleans up all resources

## Prerequisites

Before running this notebook, ensure you have:

1. A **Google Cloud project** with billing enabled
2. The following **APIs enabled** (click the links to enable):
   - [Vertex AI API](https://console.cloud.google.com/apis/api/aiplatform.googleapis.com)
   - [Artifact Registry API](https://console.cloud.google.com/apis/api/artifactregistry.googleapis.com)
   - [Cloud Build API](https://console.cloud.google.com/apis/api/cloudbuild.googleapis.com)
3. Your account has the following **IAM roles**:
   - `Vertex AI User` (or Admin)
   - `Artifact Registry Writer`
   - `Cloud Build Editor`
   - `Storage Admin` (for staging)

In [ ]:
!pip install -q google-cloud-aiplatform gTTS pydub

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab OAuth.")
else:
    print(
        "Not running in Colab.\n"
        "Make sure Application Default Credentials are set:\n"
        "  gcloud auth application-default login"
    )

In [ ]:
# ============================================================
#  CONFIGURATION — update these for your project
# ============================================================
PROJECT_ID = "your-project-id"   # @param {type:"string"}
REGION     = "us-central1"       # @param {type:"string"}

REPOSITORY = "ml-serving"        # Artifact Registry repo name
IMAGE_NAME = "parakeet-eou-triton"
IMAGE_TAG  = "v1"

# Derived — no changes needed below
CONTAINER_IMAGE_URI = (
    f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY}/{IMAGE_NAME}:{IMAGE_TAG}"
)
ENDPOINT_DISPLAY_NAME = "parakeet-eou-triton-endpoint"
MODEL_DISPLAY_NAME    = "parakeet-realtime-eou-120m-triton"

# Point gcloud at the right project
!gcloud config set project {PROJECT_ID} --quiet

# Initialise Vertex AI SDK
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION)

print(f"Project:   {PROJECT_ID}")
print(f"Region:    {REGION}")
print(f"Container: {CONTAINER_IMAGE_URI}")

## Step 1 — Build the Triton Serving Container

We create a Docker image with three components:

1. **Triton Python Backend model** (`model.py`) — loads the NeMo ASR model and handles batched inference
2. **Triton configuration** (`config.pbtxt`) — enables dynamic batching (up to 8 requests batched, 50 ms queue delay)
3. **Vertex AI compatibility proxy** (`server.py`) — thin Flask layer that translates between Vertex AI's `{"instances": [...]}` format and Triton's native v2 inference API

```
Container Architecture:
┌─────────────────────────────────────────────────┐
│  Port 8080: Flask proxy (Vertex AI interface)   │
│    GET  /health  → Triton /v2/health/ready      │
│    POST /predict → Triton /v2/models/.../infer  │
├─────────────────────────────────────────────────┤
│  Port 8000: Triton Inference Server (internal)  │
│    Model: parakeet_asr (Python Backend)         │
│    Dynamic batching: up to 8 requests           │
│    GPU instance: 1                              │
└─────────────────────────────────────────────────┘
```

In [ ]:
import os

BUILD_DIR = "/content/parakeet_triton_serving"
MODEL_DIR = os.path.join(BUILD_DIR, "models", "parakeet_asr")
os.makedirs(os.path.join(MODEL_DIR, "1"), exist_ok=True)
print(f"Build directory: {BUILD_DIR}")
print(f"Model directory: {MODEL_DIR}")

In [ ]:
%%writefile /content/parakeet_triton_serving/models/parakeet_asr/config.pbtxt
name: "parakeet_asr"
backend: "python"
max_batch_size: 8

input [
  {
    name: "AUDIO_B64"
    data_type: TYPE_STRING
    dims: [ 1 ]
  }
]

output [
  {
    name: "TRANSCRIPTION"
    data_type: TYPE_STRING
    dims: [ 1 ]
  },
  {
    name: "EOU_DETECTED"
    data_type: TYPE_BOOL
    dims: [ 1 ]
  }
]

instance_group [
  {
    kind: KIND_GPU
    count: 1
  }
]

dynamic_batching {
  preferred_batch_size: [ 4, 8 ]
  max_queue_delay_microseconds: 50000
}

In [ ]:
%%writefile /content/parakeet_triton_serving/models/parakeet_asr/1/model.py
"""
Triton Python Backend for NVIDIA Parakeet Realtime EOU 120M.

Handles batched inference requests from Triton Inference Server.
Each request contains a base64-encoded WAV audio clip; the model
returns the transcription and whether an end-of-utterance was detected.
"""

import base64
import logging
import os
import sys
import tempfile

import numpy as np
import torch

import triton_python_backend_utils as pb_utils

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("parakeet_triton")


class TritonPythonModel:
    """Triton Python Backend model for Parakeet ASR."""

    def initialize(self, args):
        """Load the NeMo ASR model. Called once when Triton loads the model."""
        logger.info("Loading NVIDIA Parakeet Realtime EOU 120M model ...")

        import nemo.collections.asr as nemo_asr

        model_name = "nvidia/parakeet_realtime_eou_120m-v1"
        self.model = nemo_asr.models.ASRModel.from_pretrained(
            model_name=model_name
        )
        self.model.eval()

        if torch.cuda.is_available():
            self.model = self.model.cuda()
            logger.info("Model loaded on GPU: %s", torch.cuda.get_device_name(0))
        else:
            logger.warning("No GPU detected — inference will be slow.")

        logger.info("Model ready for inference.")

    def execute(self, requests):
        """Handle a batch of inference requests.

        Each request contains an AUDIO_B64 input (base64-encoded WAV).
        Returns TRANSCRIPTION (string) and EOU_DETECTED (bool) for each.
        """
        responses = []

        # Collect all audio files for batch transcription
        tmp_paths = []
        valid_indices = []
        error_results = {}

        for idx, request in enumerate(requests):
            try:
                audio_b64_tensor = pb_utils.get_input_tensor_by_name(
                    request, "AUDIO_B64"
                )
                audio_b64 = audio_b64_tensor.as_numpy()[0][0]
                if isinstance(audio_b64, bytes):
                    audio_b64 = audio_b64.decode("utf-8")

                if not audio_b64:
                    error_results[idx] = "Missing audio data."
                    continue

                with tempfile.NamedTemporaryFile(
                    suffix=".wav", delete=False
                ) as f:
                    f.write(base64.b64decode(audio_b64))
                    tmp_paths.append(f.name)
                    valid_indices.append(idx)

            except Exception as exc:
                logger.error("Failed to decode request %d: %s", idx, exc)
                error_results[idx] = str(exc)

        # Batch transcription — all valid audio files at once
        transcriptions = {}
        if tmp_paths:
            try:
                outputs = self.model.transcribe(tmp_paths)
                for i, output in enumerate(outputs):
                    text = (
                        output.text
                        if hasattr(output, "text")
                        else str(output)
                    )
                    transcriptions[valid_indices[i]] = text
            except Exception as exc:
                logger.error("Batch transcription failed: %s", exc, exc_info=True)
                for i in valid_indices:
                    error_results[i] = str(exc)
            finally:
                for p in tmp_paths:
                    try:
                        os.unlink(p)
                    except OSError:
                        pass

        # Build responses for each request
        for idx in range(len(requests)):
            if idx in transcriptions:
                text = transcriptions[idx]
                eou = "<EOU>" in text or "<eou>" in text
            elif idx in error_results:
                text = ""
                eou = False
            else:
                text = ""
                eou = False

            transcription_tensor = pb_utils.Tensor(
                "TRANSCRIPTION",
                np.array([[text]], dtype=object),
            )
            eou_tensor = pb_utils.Tensor(
                "EOU_DETECTED",
                np.array([[eou]], dtype=bool),
            )

            response = pb_utils.InferenceResponse(
                output_tensors=[transcription_tensor, eou_tensor]
            )
            responses.append(response)

        return responses

    def finalize(self):
        """Clean up when the model is unloaded."""
        logger.info("Parakeet ASR model unloaded.")

In [ ]:
%%writefile /content/parakeet_triton_serving/server.py
"""
Vertex AI compatibility proxy for Triton Inference Server.

Translates between Vertex AI's prediction format and Triton's v2 API:
  - GET  /health  → Triton /v2/health/ready
  - POST /predict → Triton /v2/models/parakeet_asr/infer

This keeps the same API contract as the Flask+gunicorn version
so existing tests and load testing scripts work unchanged.
"""

import json
import logging
import os
import sys
import urllib.error
import urllib.request

from flask import Flask, request, jsonify

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("triton_proxy")

app = Flask(__name__)

TRITON_URL = os.environ.get("TRITON_URL", "http://localhost:8000")
MODEL_NAME = "parakeet_asr"

AIP_HEALTH_ROUTE  = os.environ.get("AIP_HEALTH_ROUTE", "/health")
AIP_PREDICT_ROUTE = os.environ.get("AIP_PREDICT_ROUTE", "/predict")


@app.route(AIP_HEALTH_ROUTE, methods=["GET"])
def health():
    """Proxy health check to Triton."""
    try:
        req = urllib.request.Request(f"{TRITON_URL}/v2/health/ready")
        with urllib.request.urlopen(req, timeout=5) as resp:
            if resp.status == 200:
                return jsonify({"status": "healthy"}), 200
    except Exception as e:
        logger.warning("Triton health check failed: %s", e)
    return jsonify({"status": "unhealthy"}), 503


@app.route(AIP_PREDICT_ROUTE, methods=["POST"])
def predict():
    """Translate Vertex AI predict request to Triton v2 inference.

    Vertex AI format:
        {"instances": [{"audio_base64": "..."}]}

    Triton v2 format:
        {"inputs": [{"name": "AUDIO_B64", "shape": [N,1], "datatype": "BYTES", "data": [...]}]}
    """
    data = request.get_json(force=True)
    instances = data.get("instances", [])

    if not instances:
        return jsonify({"error": "No instances provided."}), 400

    # Build Triton batch request — one inference call for all instances
    audio_data = []
    for inst in instances:
        audio_b64 = inst.get("audio_base64", "")
        audio_data.append(audio_b64)

    triton_request = {
        "inputs": [
            {
                "name": "AUDIO_B64",
                "shape": [len(audio_data), 1],
                "datatype": "BYTES",
                "data": audio_data,
            }
        ],
        "outputs": [
            {"name": "TRANSCRIPTION"},
            {"name": "EOU_DETECTED"},
        ],
    }

    # Send to Triton
    triton_url = f"{TRITON_URL}/v2/models/{MODEL_NAME}/infer"
    triton_payload = json.dumps(triton_request).encode("utf-8")
    triton_req = urllib.request.Request(
        triton_url,
        data=triton_payload,
        headers={"Content-Type": "application/json"},
    )

    try:
        with urllib.request.urlopen(triton_req, timeout=300) as resp:
            triton_resp = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        logger.error("Triton inference failed: HTTP %d: %s", e.code, body[:500])
        return jsonify({"error": f"Triton inference failed: {body[:200]}"}), 502
    except Exception as e:
        logger.error("Triton inference failed: %s", e)
        return jsonify({"error": str(e)}), 502

    # Parse Triton response into Vertex AI format
    predictions = []
    transcriptions = []
    eou_flags = []

    for output in triton_resp.get("outputs", []):
        if output["name"] == "TRANSCRIPTION":
            transcriptions = output["data"]
        elif output["name"] == "EOU_DETECTED":
            eou_flags = output["data"]

    for i in range(len(instances)):
        text = transcriptions[i] if i < len(transcriptions) else ""
        eou = eou_flags[i] if i < len(eou_flags) else False
        predictions.append({
            "transcription": text,
            "end_of_utterance_detected": bool(eou),
        })

    return jsonify({"predictions": predictions})


if __name__ == "__main__":
    port = int(os.environ.get("AIP_HTTP_PORT", 8080))
    app.run(host="0.0.0.0", port=port)

In [ ]:
%%writefile /content/parakeet_triton_serving/Dockerfile
FROM nvcr.io/nvidia/tritonserver:24.01-py3

# System deps for audio processing
RUN apt-get update && \
    apt-get install -y --no-install-recommends libsndfile1 ffmpeg sox && \
    rm -rf /var/lib/apt/lists/*

# The tritonserver base image has a distutils-installed blinker that pip
# cannot uninstall. Remove it manually so Flask can install its version.
RUN rm -rf /usr/lib/python3/dist-packages/blinker* \
           /usr/lib/python3/dist-packages/blinker-*.egg-info

# Python deps — NeMo ASR toolkit + proxy server
# NeMo pins its own PyTorch version, so install it first.
RUN pip install --no-cache-dir \
    Cython packaging \
    flask gunicorn soundfile \
    "nemo_toolkit[asr]"

# Install torchvision and torchaudio AFTER NeMo so pip resolves versions
# that match the PyTorch version NeMo installed.
# Pin setuptools<78 because 78+ removed pkg_resources, which NeMo needs.
RUN pip install --no-cache-dir torchvision torchaudio "setuptools<78"

# Pre-download the model during build so startup is fast
RUN python3 -c "\
import nemo.collections.asr as nemo_asr; \
m = nemo_asr.models.ASRModel.from_pretrained( \
    'nvidia/parakeet_realtime_eou_120m-v1'); \
del m; print('Model cached.')"

# Copy Triton model repository
COPY models/ /models/

# Copy Vertex AI compatibility proxy
COPY server.py /app/server.py
WORKDIR /app

# Vertex AI environment variables
ENV AIP_HTTP_PORT=8080
ENV AIP_HEALTH_ROUTE=/health
ENV AIP_PREDICT_ROUTE=/predict
ENV TRITON_URL=http://localhost:8000

EXPOSE 8080

# Clear the base image's ENTRYPOINT so our shell-form CMD runs correctly.
# The tritonserver image sets ENTRYPOINT ["tritonserver"], which would
# cause the shell wrapper to be passed as arguments to tritonserver.
ENTRYPOINT []

# Start Triton in the background, then the compatibility proxy
CMD tritonserver --model-repository=/models --http-port=8000 --log-verbose=0 & \
    TRITON_PID=$! && \
    echo "Waiting for Triton to start ..." && \
    for i in $(seq 1 120); do \
        if curl -sf http://localhost:8000/v2/health/ready > /dev/null 2>&1; then \
            echo "Triton is ready."; break; \
        fi; \
        sleep 2; \
    done && \
    exec gunicorn \
        --bind 0.0.0.0:${AIP_HTTP_PORT} \
        --timeout 300 \
        --workers 1 \
        --threads 4 \
        server:app

In [ ]:
# Create the Artifact Registry repository (idempotent)
!gcloud artifacts repositories create {REPOSITORY} \
    --repository-format=docker \
    --location={REGION} \
    --description="ML model serving containers" \
    --quiet 2>/dev/null || true

# Build & push the container image via Cloud Build.
# This typically takes 20-40 minutes (NeMo install + model download).
print(f"Building container: {CONTAINER_IMAGE_URI}")
print("This will take 20-40 min — grab a coffee.\n")

!cd /content/parakeet_triton_serving && gcloud builds submit \
    --tag {CONTAINER_IMAGE_URI} \
    --timeout=3600 \
    --machine-type=e2-highcpu-8

## Step 2 — Deploy to Vertex AI

We register the container as a Vertex AI **Model**, create an **Endpoint**,
and deploy the model on an **NVIDIA T4 GPU** (`n1-standard-4`).

The container runs two processes:
- **Triton Inference Server** on port 8000 (internal) — handles model inference with dynamic batching
- **Flask compatibility proxy** on port 8080 (exposed) — translates Vertex AI request format to Triton's v2 API

The API contract is identical to the Flask+gunicorn version, so existing tests and load testing scripts work without changes.

In [ ]:
from google.cloud import aiplatform

# Upload model to Vertex AI Model Registry
model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    description=(
        "NVIDIA Parakeet Realtime EOU 120M — "
        "streaming ASR with end-of-utterance detection "
        "(Triton Inference Server + dynamic batching)"
    ),
    serving_container_image_uri=CONTAINER_IMAGE_URI,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
    sync=True,
)
model.wait()
print(f"Model uploaded: {model.resource_name}")

# Create an endpoint
endpoint = aiplatform.Endpoint.create(
    display_name=ENDPOINT_DISPLAY_NAME,
)
print(f"Endpoint created: {endpoint.resource_name}")

# Deploy model to endpoint
print("\nDeploying model — this typically takes 10-15 minutes ...")
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name=MODEL_DISPLAY_NAME,
    machine_type="n1-standard-4",
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
    deploy_request_timeout=3600,
    sync=True,
)

print(f"\nModel deployed!")
print(f"Endpoint: {endpoint.resource_name}")

## Step 3 — Test the Deployed Model

We generate speech audio using Google Text-to-Speech, convert it to the
format the model expects (16 kHz, mono, WAV), and send it to the endpoint.

The API is identical to the Flask+gunicorn version — same request/response format.

In [ ]:
import base64
from gtts import gTTS
from pydub import AudioSegment

def text_to_base64_wav(text: str, path: str = "/tmp/tts.wav") -> str:
    """Convert text to a base64-encoded 16 kHz mono WAV via gTTS."""
    tts = gTTS(text, lang="en")
    mp3_path = path.replace(".wav", ".mp3")
    tts.save(mp3_path)
    audio = (
        AudioSegment.from_mp3(mp3_path)
        .set_frame_rate(16000)
        .set_channels(1)
        .set_sample_width(2)  # 16-bit
    )
    audio.export(path, format="wav")
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Single-utterance test
test_text = "Hello, what is the weather like today?"
audio_b64 = text_to_base64_wav(test_text)
print(f'Test audio generated: "{test_text}"')

In [ ]:
# Send to the deployed endpoint
response = endpoint.predict(instances=[{"audio_base64": audio_b64}])

print("=" * 60)
print("PREDICTION RESULT")
print("=" * 60)
for pred in response.predictions:
    print(f"  Transcription : {pred['transcription']}")
    print(f"  EOU detected  : {pred['end_of_utterance_detected']}")

In [ ]:
# Batch test with multiple utterances
test_phrases = [
    "My name is John and I work at a technology company.",
    "Can you book me a flight to New York?",
    "The quick brown fox jumps over the lazy dog.",
]

instances = [
    {"audio_base64": text_to_base64_wav(p, f"/tmp/tts_{i}.wav")}
    for i, p in enumerate(test_phrases)
]

response = endpoint.predict(instances=instances)

print("=" * 60)
print("BATCH PREDICTION RESULTS")
print("=" * 60)
for phrase, pred in zip(test_phrases, response.predictions):
    print(f'\n  Input:         "{phrase}"')
    print(f'  Transcription: "{pred[\'transcription\']}"')
    print(f"  EOU detected:  {pred['end_of_utterance_detected']}")

### (Optional) Test with your own audio

Upload a **WAV file** (16 kHz, mono recommended) and send it to the endpoint.

In [ ]:
import base64

# Upload a WAV file from your local machine
from google.colab import files
uploaded = files.upload()  # pick a .wav file

for filename, content in uploaded.items():
    audio_b64 = base64.b64encode(content).decode("utf-8")
    response = endpoint.predict(instances=[{"audio_base64": audio_b64}])
    for pred in response.predictions:
        print(f"File: {filename}")
        print(f"  Transcription: {pred['transcription']}")
        print(f"  EOU detected:  {pred['end_of_utterance_detected']}")

## Step 4 — Clean Up Resources

**Important:** The deployed endpoint incurs charges (~\$0.35/hr for the T4
node). Run the cell below to delete everything when you are done.

In [ ]:
# Undeploy model from the endpoint
endpoint.undeploy_all()
print("Model undeployed.")

# Delete the endpoint
endpoint.delete()
print("Endpoint deleted.")

# Delete the model from the registry
model.delete()
print("Model deleted.")

print("\nAll Vertex AI resources cleaned up — no further charges.")

# (Optional) Delete the container image from Artifact Registry:
# !gcloud artifacts docker images delete {CONTAINER_IMAGE_URI} --quiet